In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
data=pd.read_csv("titanic.csv")

In [20]:
df=data

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [22]:
df=df.drop(["PassengerId","Name","Cabin","Ticket"],axis=1)

Dropped PassengerId, Name, Cabin, and Ticket because they were either identifiers or less useful/highly sparse features for this model.          
               
More specifically:              
                     
PassengerId → just a unique identifier, no predictive value            
Name → mostly text data; could contain useful signals (titles like Mr/Mrs), but I didn’t do feature extraction, so removed it                      
Cabin → too many missing values (~77% missing), making it less practical for this simple model                  
Ticket → mostly unique/alphanumeric values, hard to use directly without feature engineering

In [23]:
df["Age"]=df["Age"].fillna(df["Age"].median())
df["Embarked"]=df["Embarked"].fillna(df["Embarked"].mode()[0])


In [24]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [25]:
df["Sex"]=df["Sex"].map({"male":0,"female":1})
df=pd.get_dummies(df,columns=["Embarked"],drop_first=True)

In [26]:
X=df.drop(["Survived"],axis=1)
Y=df["Survived"]

# **Model 1: KNN**

In [27]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.20,random_state=42)

In [28]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train) #scaling the entire dataframe
X_test=scaler.transform(X_test)

In [29]:
from sklearn.neighbors import KNeighborsClassifier
model=KNeighborsClassifier(n_neighbors=5) #we have choose a value of K (number of neighbors)
model.fit(X_train,Y_train)
Y_predictions=model.predict(X_test)

In [30]:
Y_predictions

array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 1, 1])

In [31]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
Acc=accuracy_score(Y_test,Y_predictions)
con=confusion_matrix(Y_test,Y_predictions)
report=classification_report(Y_test,Y_predictions)
print(Acc)
print(con)
print(report)

0.8044692737430168
[[90 15]
 [20 54]]
              precision    recall  f1-score   support

           0       0.82      0.86      0.84       105
           1       0.78      0.73      0.76        74

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.80      0.80      0.80       179



# **Model 2: Gaussian Naive Bayes**

Naive Bayes usually does NOT require feature scaling

In [32]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.20,random_state=42)

In [33]:
from sklearn.naive_bayes import GaussianNB
model=GaussianNB()
model.fit(X_train,Y_train)
Y_predictions2=model.predict(X_test)

In [34]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
Acc=accuracy_score(Y_test,Y_predictions2)
con=confusion_matrix(Y_test,Y_predictions2)
report=classification_report(Y_test,Y_predictions2)
print(Acc)
print(con)
print(report)

0.770949720670391
[[84 21]
 [20 54]]
              precision    recall  f1-score   support

           0       0.81      0.80      0.80       105
           1       0.72      0.73      0.72        74

    accuracy                           0.77       179
   macro avg       0.76      0.76      0.76       179
weighted avg       0.77      0.77      0.77       179

